In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

import pandas as pd
import numpy as np
import time

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
def load_dataset(img_size=32):
    df = pd.read_csv("Data/train.csv")

    y = df["label"].values
    X = df.drop("label", axis=1).values

    X = X / 255.0
    X = X.reshape(-1, 1, 28, 28)

    X_tensor = torch.tensor(X, dtype=torch.float32)

    # Resize dynamically
    X_tensor = F.interpolate(X_tensor, size=(img_size, img_size))

    y_tensor = torch.tensor(y, dtype=torch.long)

    dataset = TensorDataset(X_tensor, y_tensor)

    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size

    return random_split(dataset, [train_size, val_size])

In [4]:
class DenseLayer(nn.Module):
    def __init__(self, in_channels, growth_rate):
        super().__init__()
        
        self.bn = nn.BatchNorm2d(in_channels)
        self.conv = nn.Conv2d(in_channels, growth_rate, kernel_size=3, padding=1)
        
    def forward(self, x):
        out = self.conv(F.relu(self.bn(x)))
        return torch.cat([x, out], dim=1)

In [5]:
class DenseBlock(nn.Module):
    def __init__(self, num_layers, in_channels, growth_rate):
        super().__init__()
        
        layers = []
        for i in range(num_layers):
            layers.append(DenseLayer(in_channels + i * growth_rate, growth_rate))
        
        self.block = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.block(x)

In [6]:
class TransitionLayer(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        
        self.bn = nn.BatchNorm2d(in_channels)
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.pool = nn.AvgPool2d(2)  # IMPORTANT
        
    def forward(self, x):
        x = self.conv(F.relu(self.bn(x)))
        x = self.pool(x)
        return x

In [7]:
class DenseNet(nn.Module):
    def __init__(self, growth_rate=16):
        super().__init__()
        
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        
        self.block1 = DenseBlock(4, 32, growth_rate)
        self.trans1 = TransitionLayer(32 + 4*growth_rate, 64)
        
        self.block2 = DenseBlock(4, 64, growth_rate)
        
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(64 + 4*growth_rate, 10)
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.block1(x)
        x = self.trans1(x)
        x = self.block2(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

In [8]:
def train_model(model, train_dataset, val_dataset, epochs=3, batch_size=64):
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    
    start_time = time.time()
    
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
    
    end_time = time.time()
    
    if device.type == "cuda":
        memory = torch.cuda.max_memory_allocated() / (1024**2)
    else:
        memory = "CPU"
    
    return end_time - start_time, memory

In [9]:
sizes = [28, 32, 64, 96]

for s in sizes:
    train_ds, val_ds = load_dataset(img_size=s)
    model = DenseNet()
    
    time_taken, memory_used = train_model(model, train_ds, val_ds)
    
    print(f"\nInput Size: {s}x{s}")
    print("Training Time:", time_taken)
    print("GPU Memory (MB):", memory_used)


Input Size: 28x28
Training Time: 19.757589101791382
GPU Memory (MB): 206.44140625

Input Size: 32x32
Training Time: 26.800232648849487
GPU Memory (MB): 261.546875

Input Size: 64x64
Training Time: 124.7469208240509
GPU Memory (MB): 987.421875

Input Size: 96x96
Training Time: 303.78765201568604
GPU Memory (MB): 2420.67138671875
